In [1]:
import socketio
import eventlet
import numpy as np
from flask import Flask
from keras.models import load_model
import base64
from io import BytesIO
from PIL import Image               #PIL - Python Imaging library (Pillow)
import cv2
 
sio = socketio.Server()                     #Initializing the web server
 
app = Flask(__name__) #'__main__'
speed_limit = 10
def img_preprocess(img):
    img = img[60:135,:,:]
    img = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
    img = cv2.GaussianBlur(img,  (3, 3), 0)
    img = cv2.resize(img, (200, 66))
    img = img/255
    return img
 
 
@sio.on('telemetry')
def telemetry(sid, data):               #Session Id, data received from the simulator
    speed = float(data['speed'])

    image = Image.open(BytesIO(base64.b64decode(data['image'])))       
    #Decoding base 64 image,ByteIo -  buffer module to mimic our data as normal file; futher used for processing
    image = np.asarray(image)
    image = img_preprocess(image)
    image = np.array([image])           #Model requires 4D array(while our model is 3D array)
    steering_angle = float(model.predict(image))
    
    throttle = 1.0 - speed/speed_limit
    print('{} {} {}'.format(steering_angle, throttle, speed))
    send_control(steering_angle, throttle)
 
 
 
@sio.on('connect') #connect,disconnet,message
#Once connec with client need to fire an - "Event handler"
def connect(sid, environ):          #Session ID, Environment
    print('Connected')
    send_control(0, 0)
 
def send_control(steering_angle, throttle):
    sio.emit('steer', data = {                          #custom events
        'steering_angle': steering_angle.__str__(),
        'throttle': throttle.__str__()
    })
 
 
if __name__ == '__main__':
    model = load_model('model.h5')
    app = socketio.Middleware(sio, app)                     #Middleware
    eventlet.wsgi.server(eventlet.listen(('', 4567)), app)  
#WSGI, Tuple of IP(''-to listen to listen on any available IP addresses) and Port(4567),app-to which req are sent

(2708) wsgi starting up on http://0.0.0.0:4567
(2708) accepted ('127.0.0.1', 50289)


Connected
-0.1068505272269249 0.99999 0.0001
-0.1068505272269249 0.99999 0.0001
-0.10683748126029968 0.99999 0.0001
-0.10758817940950394 0.98082 0.1918
-0.11613083630800247 0.86269 1.3731
-0.1283479928970337 0.72982 2.7018
-0.11200913041830063 0.63823 3.6177
-0.13996058702468872 0.50292 4.9708
-0.13461917638778687 0.43587999999999993 5.6412
-0.08304542303085327 0.33882999999999996 6.6117
-0.10364373028278351 0.29486999999999997 7.0513
-0.1329411268234253 0.25195 7.4805
-0.11894340813159943 0.20819 7.9181
-0.09828899800777435 0.18577999999999995 8.1422
-0.08947930485010147 0.15921000000000007 8.4079
-0.05229588598012924 0.14565000000000006 8.5435
-0.017027748748660088 0.12958000000000003 8.7042
-0.04697285592556 0.12104999999999999 8.7895
-0.0039501190185546875 0.11304000000000003 8.8696
-0.012703020125627518 0.10770000000000002 8.923
0.011476438492536545 0.10261999999999993 8.9738
0.03000326082110405 0.09841999999999995 9.0158
-0.006015685386955738 0.08190000000000008 9.181
0.003476535

(2708) accepted ('127.0.0.1', 61102)


Connected
0.09983599185943604 1.0 0.0
0.09983599185943604 1.0 0.0
0.09983599185943604 1.0 0.0
0.10055810958147049 0.88865 1.1135
0.08484353125095367 0.86631 1.3369
0.06684883683919907 0.63671 3.6329
0.12401309609413147 0.19601000000000002 8.0399
0.16867469251155853 -0.018329999999999957 10.1833
0.12084882706403732 -0.18762999999999996 11.8763
0.10710429400205612 -0.25536000000000003 12.5536
0.12754309177398682 -0.25882000000000005 12.5882
0.12527461349964142 -0.25254 12.5254
0.11640439182519913 -0.23475000000000001 12.3475
0.11591297388076782 -0.20362999999999998 12.0363
0.18293397128582 -0.17226999999999992 11.7227
0.1971168965101242 -0.11256 11.1256
0.13033229112625122 -0.08021999999999996 10.8022
0.22581413388252258 -0.016310000000000047 10.1631
0.23277653753757477 0.02668999999999999 9.7331
0.19798235595226288 0.05693999999999999 9.4306
0.18071945011615753 0.08406999999999998 9.1593
0.15877144038677216 0.11731999999999998 8.8268
0.1324705332517624 0.13768999999999987 8.6231
0.14005

wsgi exiting
127.0.0.1 - - [24/Jan/2026 20:29:38] "GET /socket.io/?EIO=4&transport=websocket HTTP/1.1" 200 0 25.765792
127.0.0.1 - - [24/Jan/2026 20:29:38] "GET /socket.io/?EIO=4&transport=websocket HTTP/1.1" 200 0 103.225636
(2708) wsgi exited, is_accepting=True
